# MoS2 + hBN epsilon=20 plotting

This notebook follows the same spirit as the CNT plotting notebook: EDOS vs energy, current traces, polarization response, screened interaction/self-energy response, and only a small convergence/sanity section. Carrier densities and full lesser/greater plots are intentionally left out because they are not the main physics story for this run.


In [ ]:
from pathlib import Path
import re
import tomllib

import numpy as np
import matplotlib.pyplot as plt

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update({
    "figure.figsize": (8.5, 4.8),
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.labelsize": 11,
    "axes.titlesize": 12,
    "legend.frameon": False,
})

ROOT = Path.cwd()
CONFIG = ROOT / "quatrex_config_mos2_hbn_epsilon_20.toml"
OUT = ROOT / "epsilon_20" / "outputs_mos2_hbn_24iter_fresh"
LOG = ROOT / "epsilon_20" / "run_24iter_fresh.log"

with CONFIG.open("rb") as f:
    cfg = tomllib.load(f)

electron_cfg = cfg["electron"]
energies = np.linspace(
    electron_cfg["energy_window_min"],
    electron_cfg["energy_window_max"],
    electron_cfg["energy_window_num"],
)

# Same conversion used in the CNT notebook.
hbar_eV_s = 6.582119569e-16
response_energies = energies - energies[0]
omega = response_energies / hbar_eV_s

def iteration_from_name(path):
    return int(path.stem.rsplit("_", 1)[1])

def files_for(stem):
    return sorted(OUT.glob(f"{stem}_*.npy"), key=iteration_from_name)

def load(stem, iteration):
    return np.load(OUT / f"{stem}_{iteration}.npy", mmap_mode="r")

def summed_trace(stem, iteration):
    return np.asarray(load(stem, iteration)).sum(axis=1)

def maxabs(stem, iteration):
    return float(np.nanmax(np.abs(load(stem, iteration))))

def plot_complex_trace(x, y, label, *, ax=None, color=None, linestyle="-"):
    if ax is None:
        ax = plt.gca()
    real_line, = ax.plot(x, np.real(y), label=f"Re({label})", linewidth=2, alpha=0.85, color=color, linestyle=linestyle)
    imag_color = color if color is not None else real_line.get_color()
    ax.plot(x, np.imag(y), label=f"Im({label})", linewidth=2, alpha=0.75, color=imag_color, linestyle="--")

available_iterations = sorted({iteration_from_name(p) for p in OUT.glob("electron_ldos_*.npy")})
compare_iterations = [i for i in [0, 11, 23] if i in available_iterations]
if not compare_iterations and available_iterations:
    compare_iterations = [available_iterations[0], available_iterations[len(available_iterations)//2], available_iterations[-1]]

print(f"Output directory: {OUT.resolve()}")
print(f"Available iterations: {available_iterations[0]}..{available_iterations[-1]} ({len(available_iterations)} total)")
print(f"Comparison iterations: {compare_iterations}")
print(f"Energy grid: {energies[0]:.3f} to {energies[-1]:.3f} eV, {len(energies)} points")


## SCBA Convergence


In [ ]:
if LOG.exists():
    text = LOG.read_text(errors="replace")
    sigma_updates = [float(x) for x in re.findall(r"Maximum Self-Energy Update: ([^\n]+)", text)]
    current_diffs = [float(x) for x in re.findall(r"Contact Current Difference: ([^\n]+)", text)]

    fig, ax = plt.subplots(1, 2, figsize=(11, 4))
    ax[0].plot(range(len(sigma_updates)), sigma_updates, marker="o")
    ax[0].set_title("SCBA self-energy update")
    ax[0].set_xlabel("Iteration")
    ax[0].set_ylabel("Maximum update")

    ax[1].plot(range(len(current_diffs)), current_diffs, marker="o", color="tab:orange")
    ax[1].set_title("Contact current difference")
    ax[1].set_xlabel("Iteration")
    ax[1].set_ylabel("Difference")

    plt.tight_layout()
    print(f"Last self-energy update: {sigma_updates[-1]:.6g}")
    print(f"Last contact current difference: {current_diffs[-1]:.6g}")
else:
    print(f"Log not found: {LOG}")


## Main Observable Trends

These max-absolute-value trends are just a compact stability check. They omit electron/hole carrier densities and focus on the quantities used in the physics plots below.


In [ ]:
trend_stems = [
    "device_current",
    "electron_ldos",
    "p_retarded_density",
    "w_greater_density",
    "sigma_retarded_density",
]

fig, axes = plt.subplots(1, len(trend_stems), figsize=(17, 3.5), sharex=True)
for ax, stem in zip(axes, trend_stems):
    xs, ys = [], []
    for f in files_for(stem):
        it = iteration_from_name(f)
        xs.append(it)
        ys.append(np.nanmax(np.abs(np.load(f, mmap_mode="r"))))
    ax.plot(xs, ys, marker="o", linewidth=1.6)
    ax.set_title(stem)
    ax.set_xlabel("Iteration")
    ax.set_ylabel("max |value|")
plt.tight_layout()


## Currents vs Energy


In [ ]:
current_stems = ["i_left", "i_right", "i_meir-wingreen", "device_current"]
fig, axes = plt.subplots(2, 2, figsize=(12, 8), sharex=True)
for ax, stem in zip(axes.ravel(), current_stems):
    for it in compare_iterations:
        y = np.asarray(load(stem, it)).squeeze()
        ax.plot(energies, np.real(y), label=f"iter {it}")
    ax.set_title(stem)
    ax.set_xlabel("Energy (eV)")
    ax.set_ylabel("Current")
    ax.legend()
plt.tight_layout()


## EDOS vs Energy

This mirrors the CNT EDOS plot, but only keeps the all-orbital trace for MoS2.


In [ ]:
plt.figure(figsize=(9, 5))
for it in compare_iterations:
    edos = summed_trace("electron_ldos", it)
    plot_complex_trace(energies, edos, f"EDOS iter {it}")
plt.xlabel("Energy (eV)")
plt.ylabel("Sum electron LDOS")
plt.title("EDOS vs Energy")
plt.legend()
plt.tight_layout()


## Retarded Polarization vs Angular Frequency

This is the polarization component worth keeping front-and-center. The lesser/greater components are checked later only as a magnitude sanity check.


In [ ]:
plt.figure(figsize=(9, 5))
for it in compare_iterations:
    p_ret = summed_trace("p_retarded_density", it)
    plot_complex_trace(response_energies, p_ret, f"P_ret iter {it}")
plt.xlabel("Response energy omega = E - E_min (eV)")
plt.ylabel("Sum retarded polarization density")
plt.title("Retarded Polarization vs Angular Frequency")
plt.legend()
plt.tight_layout()


## Screened Interaction and Self-Energy Response

Quatrex currently writes `w_lesser_density` and `w_greater_density` rather than a `w_retarded_density` file here. `w_greater_density` is the non-negligible screened-interaction output in this run; `sigma_retarded_density` tracks the retarded self-energy response.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), sharex=True)

for it in compare_iterations:
    w_greater = np.asarray(load("w_greater_density", it))
    axes[0].plot(energies, np.sum(np.abs(w_greater), axis=1), label=f"iter {it}")
axes[0].set_title("sum |W_greater density|")
axes[0].set_xlabel("Energy / frequency grid (eV)")
axes[0].set_ylabel("Sum magnitude over orbitals")
axes[0].legend()

for it in compare_iterations:
    sigma_ret = summed_trace("sigma_retarded_density", it)
    plot_complex_trace(energies, sigma_ret, f"iter {it}", ax=axes[1])
axes[1].set_title("sum Sigma_retarded density")
axes[1].set_xlabel("Energy / frequency grid (eV)")
axes[1].set_ylabel("Sum over orbitals")
axes[1].legend()

plt.tight_layout()



## Lesser/Greater Magnitude Check

This is intentionally a table instead of a plot. It answers whether the lesser/greater channels are negligible without making them part of the main plotting story.


In [ ]:
check_stems = [
    "p_lesser_density", "p_greater_density", "p_retarded_density",
    "w_lesser_density", "w_greater_density",
    "sigma_lesser_density", "sigma_greater_density", "sigma_retarded_density",
]

final_it = available_iterations[-1]
print(f"Final iteration: {final_it}")
for stem in check_stems:
    if (OUT / f"{stem}_{final_it}.npy").exists():
        print(f"{stem:24s} maxabs={maxabs(stem, final_it):.6g}")


## Environment Dielectric Response

This uses the hBN export file `epsilon_environment_inverse_retarded.npy`. The plotted quantity is the average diagonal/trace of the environment inverse dielectric matrix, not the scalar input `epsilon_r`.


In [ ]:
eps_inv_path = ROOT / "epsilon_20" / "outputs_hbn_environment" / "environment" / "epsilon_environment_inverse_retarded.npy"

if eps_inv_path.exists():
    eps_inv = np.load(eps_inv_path, mmap_mode="r")
    eps_inv_trace = np.array([np.trace(eps_inv[i]) / eps_inv.shape[1] for i in range(eps_inv.shape[0])])
    eps_inv_diag_abs_mean = np.array([np.mean(np.abs(np.diag(eps_inv[i]))) for i in range(eps_inv.shape[0])])

    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), sharex=True)
    axes[0].plot(response_energies, eps_inv_trace.real, label="Re trace avg")
    axes[0].plot(response_energies, eps_inv_trace.imag, label="Im trace avg", linestyle="--")
    axes[0].set_title("hBN environment epsilon_E^{-1}")
    axes[0].set_xlabel("Response energy omega = E - E_min (eV)")
    axes[0].set_ylabel("trace(epsilon_E^{-1}) / N")
    axes[0].legend()

    axes[1].plot(response_energies, eps_inv_diag_abs_mean, color="tab:purple")
    axes[1].set_title("mean |diag(epsilon_E^{-1})|")
    axes[1].set_xlabel("Response energy omega = E - E_min (eV)")
    axes[1].set_ylabel("Mean diagonal magnitude")

    plt.tight_layout()
    print(f"Loaded {eps_inv_path}")
    print(f"shape={eps_inv.shape}, maxabs={np.nanmax(np.abs(eps_inv)):.6g}")
else:
    print(f"Missing {eps_inv_path}")


## Final Selected Summary


In [ ]:
final_it = available_iterations[-1]
summary_stems = [
    "device_current", "i_left", "i_right", "i_meir-wingreen",
    "electron_ldos", "p_retarded_density", "w_greater_density", "sigma_retarded_density",
]
for stem in summary_stems:
    f = OUT / f"{stem}_{final_it}.npy"
    arr = np.load(f, mmap_mode="r")
    print(f"{f.name:34s} shape={str(arr.shape):12s} finite={np.isfinite(arr).all()} maxabs={np.nanmax(np.abs(arr)):.6g}")
